In [1]:
from pyiceberg.catalog.rest import RestCatalog
import logging

import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

CATALOG_URL = "http://localhost:8181/catalog"
DEMO_WAREHOUSE = "cepcan"

CLIENT_ID = "spark"
CLIENT_SECRET = "2OR3eRvYfSZzzZ16MlPd95jhLnOaLM52"

catalog = RestCatalog(
    name="my_catalog",
    warehouse=DEMO_WAREHOUSE,
    uri=CATALOG_URL,
    credential=f"{CLIENT_ID}:{CLIENT_SECRET}",
    **{"oauth2-server-uri": "http://localhost:30080/realms/iceberg/protocol/openid-connect/token", "scope": "lakekeeper"},
)


In [2]:
# Create a new namespace if it doesn't already exist
test_namespace = ("pyiceberg_namespace",)
if test_namespace not in catalog.list_namespaces():
    catalog.create_namespace(test_namespace)

In [3]:
# Write data
test_table = ("pyiceberg_namespace", "my_table")
df = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "data": ["a", "b", "c"],
    }
)
pa_df = pa.Table.from_pandas(df)

if test_table in catalog.list_tables(namespace=test_namespace):
    catalog.drop_table(test_table)

table = catalog.create_table(
    test_table,
    schema=pa_df.schema,
)

table.append(pa_df)

In [4]:
# Read data
table = catalog.load_table(test_table)
table.scan().to_pandas()

,id,data
0,1,a
1,2,b
2,3,c


## Carga otro dataset

In [6]:
import polars as pl

datos = pl.read_parquet("../data/download.parquet")
datos

Municipio,Actividad económica,UE Unidades económicas,Año Censal,diversity,ubiquity,mcp,eci,pci,density,coi,cog,rca,Entidad,A111A Producción bruta total (millones de pesos)
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64
"""001 Abalá""","""Rama 1125 Acuicultura""",0,2023,21.0,472.0,0.0,-0.664857,-0.601705,0.127475,-0.720993,0.328753,0.0,null,null
"""001 Abalá""","""Rama 1141 Pesca""",0,2023,21.0,304.0,0.0,-0.664857,-0.52837,0.106812,-0.720993,0.523825,0.0,null,null
"""001 Abalá""","""Rama 1151 Servicios relacionad…",0,2023,21.0,269.0,0.0,-0.664857,0.924394,0.074308,-0.720993,1.101945,0.0,null,null
"""001 Abalá""","""Rama 1152 Servicios relacionad…",0,2023,21.0,117.0,0.0,-0.664857,1.749909,0.044232,-0.720993,1.647409,0.0,null,null
"""001 Abalá""","""Rama 1153 Servicios relacionad…",0,2023,21.0,21.0,0.0,-0.664857,3.397893,0.026401,-0.720993,1.151537,0.0,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""570 Zimatlán de Álvarez""","""Rama 8123 Servicios funerarios…",5,2023,54.0,1206.0,1.0,0.071231,-0.64597,0.4478,0.389851,0.0,1.856816,"""20 Oaxaca""",5.815
"""570 Zimatlán de Álvarez""","""Rama 8124 Estacionamientos y p…",5,2023,54.0,336.0,0.0,0.071231,1.772837,0.242171,0.389851,1.685871,0.88256,"""20 Oaxaca""",0.19
"""570 Zimatlán de Álvarez""","""Rama 8129 Servicios de revelad…",3,2023,54.0,369.0,1.0,0.071231,1.569516,0.256098,0.389851,0.0,1.261011,"""20 Oaxaca""",null


In [7]:
# Write data
write_table = ("pyiceberg_namespace", "cdata")

pa_cdata = pa.Table.from_pandas(datos.to_pandas())

if write_table in catalog.list_tables(namespace=test_namespace):
    catalog.drop_table(write_table)

table_cdata = catalog.create_table(
    write_table,
    schema=pa_cdata.schema,
)

table_cdata.append(pa_cdata)

In [13]:
# Read data
%time
catalog.load_table(write_table).scan().to_pandas()

CPU times: user 5 μs, sys: 3 μs, total: 8 μs
Wall time: 13.8 μs


,Municipio,Actividad económica,UE Unidades económicas,Año Censal,diversity,ubiquity,mcp,eci,pci,density,coi,cog,rca,Entidad,A111A Producción bruta total (millones de pesos)
0,001 Abalá,Rama 1125 Acuicultura,0,2023,21.0,472.0,0.0,-0.664857,-0.601705,0.127475,-0.720993,0.328753,0.000000,None,NaN
1,001 Abalá,Rama 1141 Pesca,0,2023,21.0,304.0,0.0,-0.664857,-0.528370,0.106812,-0.720993,0.523825,0.000000,None,NaN
2,001 Abalá,Rama 1151 Servicios relacionados con la agricu...,0,2023,21.0,269.0,0.0,-0.664857,0.924394,0.074308,-0.720993,1.101945,0.000000,None,NaN
3,001 Abalá,Rama 1152 Servicios relacionados con la cría y...,0,2023,21.0,117.0,0.0,-0.664857,1.749909,0.044232,-0.720993,1.647409,0.000000,None,NaN
4,001 Abalá,Rama 1153 Servicios relacionados con el aprove...,0,2023,21.0,21.0,0.0,-0.664857,3.397893,0.026401,-0.720993,1.151537,0.000000,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685114,570 Zimatlán de Álvarez,Rama 8123 Servicios funerarios y administració...,5,2023,54.0,1206.0,1.0,0.071231,-0.645970,0.447800,0.389851,0.000000,1.856816,20 Oaxaca,5.815
685115,570 Zimatlán de Álvarez,Rama 8124 Estacionamientos y pensiones para ve...,5,2023,54.0,336.0,0.0,0.071231,1.772837,0.242171,0.389851,1.685871,0.882560,20 Oaxaca,0.190
685116,570 Zimatlán de Álvarez,Rama 8129 Servicios de revelado e impresión de...,3,2023,54.0,369.0,1.0,0.071231,1.569516,0.256098,0.389851,0.000000,1.261011,20 Oaxaca,NaN
685117,570 Zimatlán de Álvarez,Rama 8131 Asociaciones y organizaciones comerc...,0,2023,54.0,968.0,0.0,0.071231,-0.509147,0.387929,0.389851,0.538384,0.000000,None,NaN
